# Forge V2 - Smartphone Price Prediction


Objective

Build a production-ready machine learning pipeline for smartphone price prediction.

*Goals*
- Improve V1 model performance
- Engineer better features
- Compare multiple ML models
- Build a reusable preprocessing pipeline
- Export a production ready model

In [62]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Baseline Model
from sklearn.linear_model import LinearRegression

# Settings
pd.set_option("display.max_columns", None)

In [63]:
df = pd.read_csv("data_smartphone.csv")
df.head()

,brand_name,Name,Price,RAM,OS,storage,Battery_cap,has_fast_charging,has_fingerprints,has_nfc,has_5g,processor_brand,num_core,primery_rear_camera,Num_Rear_Cameras,primery_front_camera,num_front_camera,display_size(inch),refresh_rate(hz),display_types
0,vivo,vivo v50,34999,8.0,android,128.0,6000,Yes,Yes,No,Yes,snapdragon,8.0,50.0,2,50.0,1,6.77,120.0,amoled display
1,realme,realme p3 pro,21999,8.0,android,128.0,6000,Yes,Yes,No,Yes,snapdragon,8.0,50.0,2,16.0,1,6.83,120.0,amoled display
2,realme,realme 14 pro plus,27999,8.0,android,128.0,6000,Yes,Yes,No,Yes,snapdragon,8.0,50.0,3,32.0,1,6.83,120.0,oled display
3,samsung,samsung galaxy s25 ultra,129999,12.0,android,256.0,5000,Yes,Yes,Yes,Yes,snapdragon,8.0,200.0,4,12.0,1,6.90,120.0,amoled display
4,vivo,vivo t3 pro,22999,8.0,android,128.0,5500,Yes,Yes,No,Yes,snapdragon,8.0,50.0,2,16.0,1,6.77,120.0,amoled display


In [64]:
print(df.shape)
df.info()

(3260, 20)
<class 'pandas.DataFrame'>
RangeIndex: 3260 entries, 0 to 3259
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   brand_name            3260 non-null   str    
 1   Name                  3260 non-null   str    
 2   Price                 3260 non-null   int64  
 3   RAM                   3260 non-null   float64
 4   OS                    3260 non-null   str    
 5   storage               3260 non-null   float64
 6   Battery_cap           3260 non-null   int64  
 7   has_fast_charging     3260 non-null   str    
 8   has_fingerprints      2534 non-null   str    
 9   has_nfc               2534 non-null   str    
 10  has_5g                2534 non-null   str    
 11  processor_brand       3260 non-null   str    
 12  num_core              3085 non-null   float64
 13  primery_rear_camera   3260 non-null   float64
 14  Num_Rear_Cameras      3260 non-null   int64  
 15  primery_front_camera 

In [65]:
df["Name"].head(30)

0                         vivo v50
1                    realme p3 pro
2               realme 14 pro plus
3         samsung galaxy s25 ultra
4                      vivo t3 pro
5          motorola edge 50 fusion
6                         moto g85
7                      oneplus 13r
8                      poco x7 pro
9             oneplus nord ce 4 5g
10                        vivo v40
11           samsung galaxy m35 5g
12                      oneplus 13
13                      iqoo 13 5g
14    xiaomi redmi note 14 pro+ 5g
15                  oneplus nord 4
16           samsung galaxy f06 5g
17           samsung galaxy a35 5g
18         xiaomi redmi note 14 5g
19         motorola edge 50 pro 5g
20                   vivo x200 pro
21                       vivo v40e
22              samsung galaxy s25
23                    iqoo z9s pro
24                        iqoo z9s
25            motorola edge 50 neo
26                        vivo t3x
27        samsung galaxy s24 ultra
28                  

In [66]:
df["num_core"].info()
df["num_core"].value_counts(dropna=False)

<class 'pandas.Series'>
RangeIndex: 3260 entries, 0 to 3259
Series name: num_core
Non-Null Count  Dtype  
--------------  -----  
3085 non-null   float64
dtypes: float64(1)
memory usage: 25.6 KB


num_core
8.0     2332
4.0      613
NaN      175
6.0      103
2.0       26
10.0       9
1.0        2
Name: count, dtype: int64

In [67]:
df[df["num_core"].isna()]

,brand_name,Name,Price,RAM,OS,storage,Battery_cap,has_fast_charging,has_fingerprints,has_nfc,has_5g,processor_brand,num_core,primery_rear_camera,Num_Rear_Cameras,primery_front_camera,num_front_camera,display_size(inch),refresh_rate(hz),display_types
87,google,google pixel 8a,37999,8.00,android,128.0,4492,Yes,Yes,Yes,Yes,google,NaN,64.0,2,13.0,1,6.1,120.0,oled display
162,google,google pixel 8,49999,8.00,android,128.0,4575,Yes,Yes,Yes,Yes,google,NaN,50.0,2,10.5,1,6.2,120.0,oled display
284,google,google pixel 8 pro,101999,12.00,android,128.0,5050,Yes,Yes,Yes,Yes,google,NaN,50.0,3,10.5,1,6.7,120.0,oled display
487,Other,reliance jiophone prima 2 4g,2799,0.50,other,4.0,2000,No,NaN,NaN,NaN,snapdragon,NaN,0.3,1,0.3,1,2.4,NaN,tft display
727,google,google pixel 8a 256gb,59999,8.00,android,256.0,4492,Yes,Yes,Yes,Yes,google,NaN,64.0,2,13.0,1,6.1,120.0,oled display
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3142,panasonic,panasonic eluga l 4g,9792,1.00,android,8.0,2000,No,NaN,NaN,NaN,snapdragon,NaN,8.0,1,5.0,1,5.0,NaN,lcd display
3150,intex,intex aqua 4g strong,4999,0.75,android,4.0,1700,No,NaN,NaN,NaN,mediatek,NaN,2.0,1,0.3,1,4.0,NaN,tft display
3170,panasonic,panasonic eluga tapp,6500,2.00,android,16.0,2800,No,Yes,No,No,mediatek,NaN,8.0,1,5.0,1,5.0,NaN,lcd display
3193,panasonic,panasonic eluga i2 activ,7490,1.00,android,16.0,2200,No,NaN,NaN,NaN,mediatek,NaN,8.0,1,5.0,1,5.0,NaN,lcd display


In [68]:
df['num_core'] = df['num_core'].fillna(df['num_core'].median())
df['has_5g'] = df['has_5g'].fillna(df['has_5g'].mode()[0])
df['has_nfc'] = df['has_nfc'].fillna(df['has_nfc'].mode()[0])
df['has_fingerprints'] = df['has_fingerprints'].fillna(df['has_fingerprints'].mode()[0])

In [69]:
print(df.isna().sum())  

brand_name                 0
Name                       0
Price                      0
RAM                        0
OS                         0
storage                    0
Battery_cap                0
has_fast_charging          0
has_fingerprints           0
has_nfc                    0
has_5g                     0
processor_brand            0
num_core                   0
primery_rear_camera        0
Num_Rear_Cameras           0
primery_front_camera       0
num_front_camera           0
display_size(inch)         0
refresh_rate(hz)        1731
display_types              0
dtype: int64


In [70]:
print(f"Duplicate rows: {df.duplicated().sum()}")
df.describe().T

Duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
Price,3260.0,20181.384356,24145.388368,2500.00,7490.0,11999.000,21999.00,200999.00
RAM,3260.0,5.065874,3.256896,0.25,3.0,4.000,8.00,24.00
storage,3260.0,112.040893,126.893532,0.31,32.0,64.000,128.00,1024.00
Battery_cap,3260.0,4163.485583,1312.404904,1100.00,3007.5,4500.000,5000.00,22000.00
num_core,3260.0,7.138037,1.649559,1.00,8.0,8.000,8.00,10.00
primery_rear_camera,3260.0,32.655828,29.397695,0.30,12.0,16.000,50.00,200.00
Num_Rear_Cameras,3260.0,2.076994,0.990856,1.00,1.0,2.000,3.00,5.00
primery_front_camera,3260.0,12.555767,10.564795,0.30,5.0,8.000,16.00,60.00
num_front_camera,3260.0,1.026994,0.162090,1.00,1.0,1.000,1.00,2.00
display_size(inch),3260.0,6.097110,0.741478,2.40,5.5,6.455,6.67,8.03


In [71]:
columns_to_drop = [
    "refresh_rate(hz)",
    "Name"
]

df = df.drop(columns=columns_to_drop)

In [72]:
df = df.rename(columns={
    "Price": "price",
    "RAM": "ram",
    "OS": "os",
    "Battery_cap": "battery_capacity",
    "has_fingerprints": "has_fingerprint",
    "num_core": "num_cores",
    "primery_rear_camera": "primary_rear_camera",
    "Num_Rear_Cameras": "num_rear_cameras",
    "primery_front_camera": "primary_front_camera",
    "num_front_camera": "num_front_cameras",
    "display_size(inch)": "display_size",
    "refresh_rate(hz)": "refresh_rate",
    "display_types": "display_type"
})

In [73]:
df.to_csv("cleaned_smartphones.csv", index=False)

In [74]:
df = pd.read_csv("cleaned_smartphones.csv") 

x = df.drop("price", axis=1)
y = df["price"]

In [75]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 3260 entries, 0 to 3259
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   brand_name            3260 non-null   str    
 1   ram                   3260 non-null   float64
 2   os                    3260 non-null   str    
 3   storage               3260 non-null   float64
 4   battery_capacity      3260 non-null   int64  
 5   has_fast_charging     3260 non-null   str    
 6   has_fingerprint       3260 non-null   str    
 7   has_nfc               3260 non-null   str    
 8   has_5g                3260 non-null   str    
 9   processor_brand       3260 non-null   str    
 10  num_cores             3260 non-null   float64
 11  primary_rear_camera   3260 non-null   float64
 12  num_rear_cameras      3260 non-null   int64  
 13  primary_front_camera  3260 non-null   float64
 14  num_front_cameras     3260 non-null   int64  
 15  display_size          3260 non-n

In [76]:

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [77]:
categorical_features = x.select_dtypes(include=["object"]).columns.tolist()

numerical_features = x.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print(categorical_features)
print(numerical_features)

['brand_name', 'os', 'has_fast_charging', 'has_fingerprint', 'has_nfc', 'has_5g', 'processor_brand', 'display_type']
['ram', 'storage', 'battery_capacity', 'num_cores', 'primary_rear_camera', 'num_rear_cameras', 'primary_front_camera', 'num_front_cameras', 'display_size']


C:\Users\Ishaan\AppData\Local\Temp\ipykernel_2312\1753377603.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = x.select_dtypes(include=["object"]).columns.tolist()


In [78]:
print(x_train.shape)
print(x_test.shape)

(2608, 17)
(652, 17)


In [79]:
numerical_transformer = Pipeline(
    steps = [
        ("impputer", SimpleImputer(strategy = "median"))
    ]
)

categorical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown = "ignore"))
    ]
    
)

In [80]:
preprocessor = ColumnTransformer(
    transformers = [
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [81]:
model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("regression", LinearRegression())
    ]
)

In [82]:
model.fit(x_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['brand_name','ram','os',...,'num_front_cameras','display_size', 'display_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This sub

In [83]:
y_pred = model.predict(x_test)

In [84]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

MAE : 7120.19
RMSE: 11686.67
R²  : 0.7208
